# Lecture 7 — Class Exercise
## Part-to-Whole: Hierarchical Visualization

> 


In [1]:
import pandas as pd
import plotly.express as px
import numpy as np

# Dataset: Global Energy Mix by Country and Source
df = pd.read_csv('../data/global_energy_mix.csv')

# Source type mapping — reuse from lecture
source_category = {
    'Coal': 'Fossil', 'Oil': 'Fossil', 'Natural Gas': 'Fossil',
    'Nuclear': 'Low-carbon', 'Hydro': 'Low-carbon',
    'Wind': 'Renewable', 'Solar': 'Renewable', 'Other Renewables': 'Renewable'
}
df['Source_Type'] = df['Source'].map(source_category)

print(f"Loaded: {len(df)} rows")
print(df.head(10))


FileNotFoundError: [Errno 2] No such file or directory: '../data/global_energy_mix.csv'

## Task 1 — Treemap: fossil fuel dependency by country

**What to build:** A treemap showing **fossil fuel TWh only**, broken down by Region → Country → Source (Coal / Oil / Natural Gas).

**Requirements:**
- Filter to fossil sources only before plotting
- Use `path=['Region', 'Country', 'Source']` for the hierarchy
- Colour encodes the fossil source type (Coal / Oil / Natural Gas) with a CVD-safe palette
- Show TWh values in labels — no percentages
- Grey out parent nodes (Region and Country level)
- Insight title naming which region or country is most fossil-dependent

> 💡 `df.loc[df['Source_Type'] == 'Fossil']`


In [ ]:
# Task 1 — fossil-only treemap: Region → Country → Source

import pandas as pd
import plotly.express as px

df = pd.read_csv('../data/global_energy_mix.csv')
source_category = {
    'Coal': 'Fossil', 'Oil': 'Fossil', 'Natural Gas': 'Fossil',
    'Nuclear': 'Low-carbon', 'Hydro': 'Low-carbon',
    'Wind': 'Renewable', 'Solar': 'Renewable', 'Other Renewables': 'Renewable',
}
df['Source_Type'] = df['Source'].map(source_category)

df_fossil = df.loc[df['Source_Type'] == 'Fossil'].copy()

# CVD-safe Okabe–Ito colours for the three fossil sources
fossil_colours = {
    'Coal': '#D55E00',         # vermillion
    'Oil': '#0072B2',          # blue
    'Natural Gas': '#E69F00',  # orange/gold
}

fig = px.treemap(
    df_fossil,
    path=['Region', 'Country', 'Source'],
    values='TWh',
    color='Source',
    color_discrete_map=fossil_colours,
    labels={'TWh': 'Energy (TWh)', 'Source': 'Fossil source'},
    height=700,
    width=1100,
    template='plotly_white',
)

fig.update_traces(
    textinfo='label+value',
    texttemplate='%{label}<br>%{value:,.0f} TWh',
    hovertemplate='<b>%{label}</b><br>%{value:,.0f} TWh<extra></extra>',
)

# Grey parent (Region / Country) nodes — only leaf Source colours stay
fig.data[0].marker.colors = [
    c if c in fossil_colours.values() else '#DDDDDD'
    for c in fig.data[0].marker.colors
]

fig.update_layout(
    title="Asia dominates fossil energy in absolute TWh — China's coal alone rivals most countries' entire fossil mix",
    font=dict(family='Arial', size=12),
)

fig.show()


## Task 2 — Treemap vs bar: low-carbon energy by country

**What to build:** Build **both** a treemap and a horizontal bar chart showing total low-carbon TWh (Nuclear + Hydro) per country. Then answer the question in a markdown cell below.

**Requirements:**
- Filter to `Source_Type == 'Low-carbon'` and aggregate TWh by country
- Treemap: single-level `path=['All', 'Country']` with a dummy root node labelled `'Low-carbon'`
- Bar chart: sorted by TWh, horizontal orientation, CVD-safe colour
- Both charts show TWh values, not percentages
- Insight title on the bar chart naming the leading country

In [ ]:
# Task 2 — low-carbon TWh: treemap + horizontal bar

import pandas as pd
import plotly.express as px

df = pd.read_csv('../data/global_energy_mix.csv')
source_category = {
    'Coal': 'Fossil', 'Oil': 'Fossil', 'Natural Gas': 'Fossil',
    'Nuclear': 'Low-carbon', 'Hydro': 'Low-carbon',
    'Wind': 'Renewable', 'Solar': 'Renewable', 'Other Renewables': 'Renewable',
}
df['Source_Type'] = df['Source'].map(source_category)

low_carbon = (
    df.loc[df['Source_Type'] == 'Low-carbon']
      .groupby('Country', as_index=False)['TWh'].sum()
)
low_carbon['All'] = 'Low-carbon'  # dummy root for a single-level treemap

# ── Treemap ───────────────────────────────────────────────────────────────────
fig_tree = px.treemap(
    low_carbon,
    path=['All', 'Country'],
    values='TWh',
    color_discrete_sequence=['#0072B2'],  # CVD-safe blue
    labels={'TWh': 'Energy (TWh)'},
    height=500,
    width=900,
    template='plotly_white',
)
fig_tree.update_traces(
    textinfo='label+value',
    texttemplate='%{label}<br>%{value:,.0f} TWh',
    hovertemplate='<b>%{label}</b><br>%{value:,.0f} TWh<extra></extra>',
)
fig_tree.update_layout(
    title='Low-carbon energy by country (treemap) — France and Norway take the largest tiles',
    font=dict(family='Arial', size=12),
)
fig_tree.show()

# ── Horizontal bar (sorted) ───────────────────────────────────────────────────
bar_order = low_carbon.sort_values('TWh', ascending=True)['Country'].tolist()
leader = low_carbon.loc[low_carbon['TWh'].idxmax(), 'Country']
leader_twh = low_carbon['TWh'].max()

fig_bar = px.bar(
    low_carbon,
    x='TWh',
    y='Country',
    orientation='h',
    category_orders={'Country': bar_order},
    color_discrete_sequence=['#0072B2'],
    text=low_carbon['TWh'].map('{:,.0f}'.format),
    labels={'TWh': 'Low-carbon energy (TWh)', 'Country': ''},
    height=520,
    template='plotly_white',
)
fig_bar.update_traces(
    textposition='outside',
    cliponaxis=False,
    hovertemplate='<b>%{y}</b><br>%{x:,.0f} TWh<extra></extra>',
)
fig_bar.update_layout(
    title=f'{leader} leads on low-carbon energy with {leader_twh:,.0f} TWh — nuclear and hydro combined',
    font=dict(family='Arial', size=12),
    plot_bgcolor='white',
    xaxis=dict(gridcolor='#EEEEEE', title='TWh'),
    yaxis=dict(showgrid=False),
    margin=dict(l=120, r=80, t=60, b=40),
    showlegend=False,
)
fig_bar.show()

### Task 2 — reflection

**Which chart tells the ranking story more clearly, and why?**

The **horizontal bar chart**. Bars share a common baseline and are sorted by TWh, so you can read the order (France → Norway → Canada → …) in one glance. The treemap is better for *relative area* inside the low-carbon total, but comparing two similar-sized tiles (e.g. Canada vs Brazil) is harder than comparing bar lengths.

## Task 3 — 100% stacked bar: which region relies most on coal?

**What to build:** A horizontal 100% stacked bar chart showing, for each **Region**, how its fossil energy splits between Coal, Oil and Natural Gas. This is the **same data as your Task 1 treemap**, visualised a second way.

**Requirements:**
- Reuse `df_fossil` from Task 1; aggregate TWh by `Region` × `Source` and convert to **shares of each region's fossil total**
- Horizontal bars, regions **sorted by coal share** (highest at the top)
- Coal is the segment on the baseline (first in `category_orders`) and the **only coloured segment** — keep it the same colour as in Task 1; Oil and Natural Gas in two greys
- Percentage labels on the coal segments only
- Insight title naming the most coal-dependent region and its coal share

> 💡 `df.groupby('Region')['TWh'].transform('sum')` gives each row its region total

In [ ]:
# Task 3 — 100% stacked bar: coal / oil / gas mix of each region's fossil energy

import numpy as np
import pandas as pd
import plotly.express as px

df = pd.read_csv('../data/global_energy_mix.csv')
source_category = {
    'Coal': 'Fossil', 'Oil': 'Fossil', 'Natural Gas': 'Fossil',
    'Nuclear': 'Low-carbon', 'Hydro': 'Low-carbon',
    'Wind': 'Renewable', 'Solar': 'Renewable', 'Other Renewables': 'Renewable',
}
df['Source_Type'] = df['Source'].map(source_category)
df_fossil = df.loc[df['Source_Type'] == 'Fossil'].copy()  # same filter as Task 1

# Aggregate Region × Source, then convert to share of each region's fossil total
mix = df_fossil.groupby(['Region', 'Source'], as_index=False)['TWh'].sum()
mix['Share'] = mix['TWh'] / mix.groupby('Region')['TWh'].transform('sum')

# Sort regions by coal share — highest at the top
region_order = (
    mix.loc[mix['Source'] == 'Coal']
       .sort_values('Share', ascending=False)['Region']
       .tolist()
)
# Regions with no coal (e.g. Middle East) still appear, at the bottom
for r in mix['Region'].unique():
    if r not in region_order:
        region_order.append(r)

# Same coal colour as Task 1; Oil and Natural Gas in grey
accent_mapping = {
    'Coal': '#D55E00',
    'Oil': '#BBBBBB',
    'Natural Gas': '#DDDDDD',
}

mix['Label'] = np.where(
    mix['Source'] == 'Coal',
    mix['Share'].map('{:.0%}'.format),
    '',
)

top = mix.loc[mix['Source'] == 'Coal'].sort_values('Share', ascending=False).iloc[0]

fig = px.bar(
    data_frame=mix,
    x='Share',
    y='Region',
    color='Source',
    text='Label',
    color_discrete_map=accent_mapping,
    category_orders={
        'Region': region_order,
        'Source': ['Coal', 'Oil', 'Natural Gas'],  # coal on the baseline
    },
    labels={'Share': 'Share of regional fossil energy', 'Source': ''},
    hover_name='Region',
    hover_data={'Label': False, 'Share': ':.1%', 'TWh': ':,.0f'},
    height=480,
    width=900,
    template='plotly_white',
)

fig.update_traces(
    textposition='inside',
    insidetextanchor='start',
    textfont=dict(color='white', size=12),
    marker_line_color='white',
    marker_line_width=1,
)

fig.update_layout(
    title=f"{top['Region']} relies most on coal within its fossil mix — {top['Share']:.0%} of fossil energy is coal",
    font=dict(family='Arial', size=12),
    bargap=0.3,
    legend=dict(orientation='h', x=0, y=1.08, title=''),
    xaxis=dict(visible=False, range=[0, 1]),
    yaxis=dict(title=''),
)

fig.add_annotation(
    x=1, y=0, xref='x', yref='paper',
    xanchor='right', yanchor='top', yshift=-4,
    text='100%',
    showarrow=False,
    font=dict(size=11, color='grey'),
)

fig.show()

### Task 3 — reflection

**Your Task 1 treemap and this stacked bar show the same fossil data. Name one question each chart answers that the other cannot.**

- **Treemap:** *Where is fossil energy consumed in absolute TWh, and how is it nested (region → country → source)?* You see that Asia’s rectangles dwarf Oceania, and that China’s coal tile is huge — magnitude within the hierarchy.
- **100% stacked bar:** *Within each region’s fossil mix, what share is coal vs oil vs gas?* Every bar is scaled to 100%, so Africa’s ~78% coal share is obvious even though Africa’s absolute fossil TWh is much smaller than Asia’s. The treemap hides that mix comparison because tile area encodes totals, not shares.